# FLRRC v1.4 training (TinyLlama 1.1B + LoRA, Colab T4)

Run cells top to bottom. **Runtime -> Change runtime type -> T4 GPU first.**
Changes vs v1.3: 75 reviewed samples, answer ends with `</s>`, distinct pad token,
learning rate 2e-4 and ~95 steps (v1.3 ran only 39 steps at the default LR, loss 2.54 -> 2.34).

In [ ]:
BRANCH = "feature/react-frontend-v2.0"   # change to "dev" once the dataset PR is merged

!git clone https://github.com/AdhimulamBhargavSaiViswanath-05/flrrc-decision-engine.git
%cd flrrc-decision-engine
!git checkout {BRANCH}
!git pull origin {BRANCH}
!wc -l data/training/chatml-v1.4.jsonl

The line above must print **75**. If the file is missing, the dataset commit is not pushed to this branch yet.

In [ ]:
!pip install -q -U transformers peft accelerate trl datasets bitsandbytes

If the next cell shows an import error, use **Runtime -> Restart session** and run again from this cell. Do NOT downgrade trl/transformers.

In [ ]:
import inspect, torch, transformers, peft, trl
from trl import SFTTrainer, SFTConfig

print("GPU:", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)

cfg_params = inspect.signature(SFTConfig.__init__).parameters
trn_params = inspect.signature(SFTTrainer.__init__).parameters
print("SFTConfig has max_length:", "max_length" in cfg_params, "| max_seq_length:", "max_seq_length" in cfg_params)
print("SFTTrainer has processing_class:", "processing_class" in trn_params, "| tokenizer:", "tokenizer" in trn_params)

In [ ]:
# T4 fix used in v1.3 (only exists in newer trl, so it is guarded)
try:
    import trl.kernels.chunked_logprob as _k
    def _patched_addmm(acc, a, b):
        acc.addmm_(a.float(), b.float())
    _k._addmm_fp32 = _patched_addmm
    print("Patched T4 kernel")
except Exception as e:
    print("No kernel patch needed:", type(e).__name__)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.unk_token   # NOT eos, so the </s> label is never masked as padding
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/training/chatml-v1.4.jsonl")["train"]
print("samples:", len(dataset))
SYSTEM_PROMPT = dataset[0]["messages"][0]["content"]
print("system prompt:", SYSTEM_PROMPT)

def fmt(ex):
    # Same layout as v1.3 (<|role|>\ncontent\n), plus </s> after the assistant answer so the model learns to stop.
    txt = ""
    for m in ex["messages"]:
        if m["role"] == "assistant":
            txt += f"<|assistant|>\n{m['content']}</s>"
        else:
            txt += f"<|{m['role']}|>\n{m['content']}\n"
    return txt

sample_txt = fmt(dataset[0])
ids = tokenizer(sample_txt).input_ids
print("tokens in sample 0:", len(ids))
print("last 6 tokens:", tokenizer.convert_ids_to_tokens(ids[-6:]))
print("max tokens over dataset:", max(len(tokenizer(fmt(e)).input_ids) for e in dataset))

Check the output above: last tokens should end with the JSON closing `}` then `</s>` (a single `</s>`). Max tokens must be below 1024.

In [ ]:
cfg = dict(
    output_dir="models/flrrc-v1.4-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    packing=False,
    fp16=False,
    bf16=False,
)
if "max_length" in cfg_params:
    cfg["max_length"] = 1024
elif "max_seq_length" in cfg_params:
    cfg["max_seq_length"] = 1024

config = SFTConfig(**cfg)

tok_kw = "processing_class" if "processing_class" in trn_params else "tokenizer"
trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=fmt,
    **{tok_kw: tokenizer},
)
print("Trainer ready | tokenizer kwarg:", tok_kw, "| expected steps ~", (len(dataset) // 4 + 1) * 5)

In [ ]:
trainer.train()
for row in trainer.state.log_history:
    if "loss" in row:
        print(f"step {row['step']:>4}  loss {row['loss']:.4f}")

Compare with v1.3: loss went 2.54 -> 2.34 in 39 steps. For v1.4 the loss should fall much further. Paste the loss table back before continuing.

In [ ]:
trainer.model.save_pretrained("models/flrrc-v1.4-lora-final")
tokenizer.save_pretrained("models/flrrc-v1.4-lora-final")
!ls -lh models/flrrc-v1.4-lora-final/

In [ ]:
import json, re

trainer.model.config.use_cache = True
trainer.model.gradient_checkpointing_disable()
trainer.model.eval()

def build_prompt(question):
    return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\nQuestion: {question}\n<|assistant|>\n"

tests = [
    "Gym ki vellala? (energy 3/10, sleep 5 hrs, exam repu)",
    "I have a job offer at 18 LPA from a startup and my MNC pays 12 LPA. Should I switch?",
    "College ki vellala vadda? (attendance 68%, lab today, energy 4/10)",
    "Should I move to Bangalore?",
]

for q in tests:
    inputs = tokenizer(build_prompt(q), return_tensors="pt").to(trainer.model.device)
    out = trainer.model.generate(**inputs, max_new_tokens=800, do_sample=True, temperature=0.7, top_p=0.9,
                                 repetition_penalty=1.05, eos_token_id=tokenizer.eos_token_id,
                                 pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    try:
        data = json.loads(text)
        ok = all(k in data for k in ["box_1_flrrc","box_2_equation","box_3_graph","box_4_risk","box_5_next_action"])
        status = f"VALID JSON, 5 boxes: {ok}"
    except Exception as e:
        status = f"NOT valid JSON ({type(e).__name__})"
    print("=" * 70)
    print("Q:", q)
    print("->", status)
    print(text[:1200])

In [ ]:
!zip -r flrrc-v1.4-lora.zip models/flrrc-v1.4-lora-final -q
!ls -lh flrrc-v1.4-lora.zip
from google.colab import files
files.download("flrrc-v1.4-lora.zip")